# Gender-Differentiated Consumer Price Index (G-CPI)
## ENIGH Estacional 2022 — Expenditure Weight Construction and Inflation Gap Analysis

**Data source:** INEGI Encuesta Nacional de Ingresos y Gastos de los Hogares (ENIGH) Estacional 2022  
**Price index:** Índice Nacional de Precios al Consumidor (INPC) — CCIF 2018 classification  
**Objective:** Construct gender-differentiated expenditure weights by household headship type and compute the inflation gap between male-headed and female-headed households.

---


## Section 1 — Environment Setup and Data Loading

All input files are expected in the current working directory. Output files are saved to `./outputs/`.


In [ ]:
# ── 1.1  Imports and directory setup ────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
import os

DATA_DIR = Path(".")        # ENIGH CSV files are located here
OUT_DIR  = Path("./outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Working directory:", os.getcwd())
print("\nCSV files found:")
for f in sorted(Path(".").glob("*.csv")):
    print(f"  {f.name}")


In [ ]:
# ── 1.2  Robust CSV reader (handles latin-1 and UTF-8 encodings) ─────────────
def read_csv_robust(path):
    """
    Attempts to read a CSV file with multiple encodings commonly used
    in INEGI microdata releases (UTF-8, latin-1, cp1252).
    """
    for enc in ["utf-8", "latin-1", "cp1252"]:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path, encoding="latin-1", errors="replace", low_memory=False)

df_hog = read_csv_robust("hogares.csv")
df_pob = read_csv_robust("poblacion.csv")
df_gh  = read_csv_robust("gastoshogar.csv")
df_gp  = read_csv_robust("gastospersona.csv")

print("Dimensions of loaded datasets:")
print(f"  hogares       : {df_hog.shape[0]:,} rows × {df_hog.shape[1]} columns")
print(f"  poblacion     : {df_pob.shape[0]:,} rows × {df_pob.shape[1]} columns")
print(f"  gastoshogar   : {df_gh.shape[0]:,} rows × {df_gh.shape[1]} columns")
print(f"  gastospersona : {df_gp.shape[0]:,} rows × {df_gp.shape[1]} columns")


## Section 2 — Household Identifier and Headship Classification

A unique household identifier (`IDHOG`) is constructed by concatenating the dwelling folio (`folioviv`) and household folio (`foliohog`). This key is standardized across all microdata files to enable consistent merges.

Households are then classified into four headship types based on the sex of the household head and the presence of a partner, following the ENIGH variable `parentesco == 101` (household head):

| Type | Description |
|------|-------------|
| `JEFATURA_HOMBRE_CON_PAREJA` | Male head, partner present |
| `JEFATURA_HOMBRE_SIN_PAREJA` | Male head, no partner |
| `JEFATURA_MUJER_CON_PAREJA` | Female head, partner present |
| `JEFATURA_MUJER_SIN_PAREJA` | Female head, no partner |


In [ ]:
# ── 2.1  Create standardized IDHOG across all microdata files ────────────────
for df in [df_hog, df_pob, df_gh, df_gp]:
    df["IDHOG"] = (
        df["folioviv"].astype(str).str.strip() + "_" +
        df["foliohog"].astype(str).str.zfill(2)
    )

print("IDHOG created. Sample values:")
print(df_hog["IDHOG"].head(5).tolist())


In [ ]:
# ── 2.2  Identify household heads and build headship typology ─────────────────
#
# parentesco == 101 identifies the household head in the ENIGH population file.
# sexo: 1 = Male, 2 = Female.
# pareja_hog == 1 indicates the presence of a partner in the household.

jefe = df_pob[df_pob["parentesco"] == 101].copy()
jefe["sexo_jefe"]    = jefe["sexo"].map({1: "HOMBRE", 2: "MUJER"})
jefe["tiene_pareja"] = (jefe["pareja_hog"].astype(str).str.strip() == "1")
jefe["tipo_jefatura"] = np.where(
    jefe["sexo_jefe"].eq("MUJER"),
    np.where(jefe["tiene_pareja"], "JEFATURA_MUJER_CON_PAREJA",  "JEFATURA_MUJER_SIN_PAREJA"),
    np.where(jefe["tiene_pareja"], "JEFATURA_HOMBRE_CON_PAREJA", "JEFATURA_HOMBRE_SIN_PAREJA")
)

headship = jefe[["IDHOG", "sexo_jefe", "tiene_pareja", "tipo_jefatura"]].drop_duplicates("IDHOG")

print(f"Households with headship information: {len(headship):,}")
print("\nDistribution by headship type:")
print(headship["tipo_jefatura"].value_counts().to_string())


## Section 3 — ENIGH–CCIF 2018 Official Crosswalk

The expenditure codes in ENIGH are mapped to the CCIF 2018 (Clasificación del Consumo Individual por Finalidades) hierarchy using the official crosswalk published by INEGI in Annex 1 of the ENIGH 2024 technical documentation (*Relación por conceptos de gasto CCIF 2018 y ENIGH 2022*).

The mapping is one-to-many: a single ENIGH code may correspond to multiple CCIF classes. The crosswalk is expanded to long format to preserve this granularity. The `ccif_div` field (first two digits of the CCIF code) is used for all division-level analyses.

Codes A243–A247 (meals consumed outside the home) are marked as *Pendiente por clasificar* in the INEGI source and are assigned to CCIF Division 11 (Restaurants and accommodation services) following standard COICOP practice.


In [ ]:
# ── 3.1  Official ENIGH–CCIF 2018 crosswalk (INEGI Annex 1) ──────────────────
#
# Source: INEGI (2024). Encuesta Nacional de Ingresos y Gastos de los Hogares
# 2024 Nueva Serie. Anexo 1: Relación por conceptos de gasto CCIF 2018 y ENIGH 2022.
# One-to-many mappings are represented as lists of CCIF class codes.

CROSSWALK_OFICIAL = {
    # GRAINS — MAIZE
    "A001": ["01.1.1.1.2"], "A002": ["01.1.1.2.1"], "A003": ["01.1.1.9.3"],
    "A004": ["01.1.1.3.1"], "A005": ["01.1.1.9.2"], "A006": ["01.1.1.9.4"],
    # GRAINS — WHEAT
    "A007": ["01.1.1.2.2"], "A008": ["01.1.1.3.2"], "A009": [],
    "A010": ["01.1.1.3.5"], "A011": ["01.1.1.3.6"], "A012": ["01.1.1.3.3"],
    "A013": ["01.1.1.3.7"], "A014": ["01.1.1.3.8"], "A015": ["01.1.1.3.4"],
    "A016": ["01.1.1.3.9"], "A017": ["01.1.1.3.A"],
    "A018": ["01.1.1.3.B", "01.1.1.5.0", "01.1.1.9.5"],
    # GRAINS — RICE AND OTHER CEREALS
    "A019": ["01.1.1.1.1"], "A020": ["01.1.1.2.3"],
    "A021": ["01.1.1.4.0"], "A022": ["01.1.1.9.1"],
    "A023": ["01.1.9.1.3"], "A024": ["01.1.1.1.3", "01.1.1.9.6"],
    # BEEF
    "A025": ["01.2.2.1"], "A026": ["01.2.2.1"], "A027": ["01.2.2.1"],
    "A028": ["01.2.2.1"], "A029": ["01.2.2.2"],
    "A030": ["01.2.2.5", "01.2.2.6", "01.2.2.7", "01.2.4.7"],
    "A065": ["01.2.2.L", "01.2.5.6"],
    # FISH AND SEAFOOD
    "A066": ["01.3.1.1"], "A067": ["01.3.1.2"], "A068": ["01.3.2.2"],
    "A069": [],
    "A070": ["01.3.2.4", "01.3.2.3", "01.3.3.1", "01.3.2.1"], "A071": [],
    "A072": ["01.3.4.1"], "A073": ["01.3.4.2"],
    "A074": ["01.3.3.2", "01.3.4.3", "01.3.5.1", "01.3.5.2", "01.3.6.0"],
    # DAIRY
    "A075": ["01.4.1.1"], "A076": ["01.4.3.1"], "A077": [],
    "A078": ["01.4.3.2"], "A079": ["01.4.2.0", "01.9.2.1"],
    "A080": ["01.4.1.2"], "A081": ["01.4.4.1", "01.4.4.2"],
    "A082": ["01.4.5.2"], "A083": ["01.4.5.5"], "A084": [],
    "A085": ["01.4.5.1"], "A086": ["01.4.5.3"], "A087": ["01.4.5.4"],
    "A088": ["01.4.5.6"], "A089": ["01.4.3.3"], "A090": ["01.5.2.0"],
    # VEGETABLES
    "A093": ["01.1.7.1.1"], "A095": ["01.1.7.1.2"],
    "A102": ["01.1.8.1.1"], "A112": ["01.1.7.6.1"],
    "A117": ["01.1.7.6.2"], "A124": ["01.1.6.1.1"],
    "A126": ["01.1.7.4.6"], "A127": ["01.1.7.2.8"],
    "A128": ["01.1.9.4.3"], "A129": ["01.1.7.2.9"],
    "A130": ["01.1.7.4.8"],
    "A131": ["01.1.7.2.A", "01.1.7.3.3", "01.1.7.4.1", "01.1.7.4.9"],
    "A132": [],
    "A133": ["01.1.7.9.4"], "A134": ["01.1.7.9.3"],
    "A135": ["01.1.7.9.8", "01.1.7.9.9"], "A136": ["01.1.7.8.0"],
    # LEGUMES
    "A137": ["01.1.7.6.1"], "A138": ["01.1.7.6.3"],
    "A139": [], "A140": ["01.1.7.6.2"], "A141": ["01.1.7.6.4"],
    "A142": ["01.1.7.7.0", "01.1.7.9.7"], "A143": [],
    # SEEDS AND FRUITS
    "A144": ["09.3.1.2.3"], "A145": ["01.1.9.4.6"], "A146": [],
    "A147": ["01.1.6.5.4"], "A148": ["01.1.6.4.1", "01.1.6.4.2"],
    "A149": ["01.1.6.3.1", "01.1.6.3.2"], "A150": [], "A151": [],
    "A152": ["01.1.6.1.2"], "A153": [], "A154": ["01.1.6.2.1"],
    "A155": ["01.1.6.2.4"], "A156": ["01.1.6.2.3"], "A157": ["01.1.6.1.3"],
    "A158": ["01.1.6.3.3"], "A159": ["01.1.6.5.1"], "A160": ["01.1.6.2.2"],
    "A173": ["01.1.6.1.3"], "A215": ["01.1.9.1.1"], "A220": ["01.2.2.1.1"],
    # CONDIMENTS AND PREPARED FOODS
    "A192": ["01.1.9.3.1", "01.1.9.3.2", "01.1.9.3.3"],
    "A193": ["01.1.9.3.7"], "A194": ["01.1.9.3.9", "01.1.9.4.5"],
    "A195": ["01.1.9.2.3"], "A196": ["01.1.9.2.2", "01.1.9.2.5"],
    "A197": ["01.1.9.2.4"],
    "A198": ["11.1.1.2.A", "01.1.9.1.1", "01.1.9.1.2"],
    "A199": ["11.1.1.2.5"], "A200": ["11.1.1.2.B"],
    "A201": ["11.1.1.2.4"], "A202": ["11.1.1.2.7"],
    "A203": ["01.1.7.4.4"], "A204": ["01.2.2.M"],
    "A205": ["01.8.9.2", "01.9.9.2"], "A206": ["01.4.7.2"],
    "A207": ["01.8.3.2", "01.8.3.3", "01.8.4.1"],
    "A208": ["01.8.6.1"], "A209": ["01.8.9.1", "01.8.9.3", "01.9.9.1"],
    "A210": [], "A211": ["01.9.9.5", "01.9.9.6", "01.3.0.0.0"],
    "A212": [], "A213": ["09.3.2.2.1"], "A242": [],
    # MEALS OUTSIDE THE HOME (INEGI: Pendiente por clasificar → CCIF 11)
    "A243": ["11.1.1.1.1"], "A244": ["11.1.1.1.1"],
    "A245": ["11.1.1.1.1"], "A246": ["11.1.1.1.1"], "A247": ["11.1.1.1.1"],
    # PUBLIC TRANSPORT
    "B001": ["07.3.1.2.1", "07.3.1.2.2", "07.3.1.1.0"],
    "B002": ["07.3.2.1.3"], "B003": ["07.3.1.2.3", "07.3.2.1.4"],
    "B004": ["07.3.2.1.5"], "B005": ["07.3.2.2.1"], "B006": ["07.3.2.1.2"],
    "B007": ["07.3.2.2.2","07.3.2.9.0","07.3.4.0.1","07.3.4.0.2",
             "07.3.5.0.0","07.3.6.0.0"],
    # HOUSEHOLD CLEANING PRODUCTS
    "C001": ["05.6.1.1.6","05.6.1.1.7"], "C002": ["05.6.1.1.A"],
    "C003": ["05.6.1.1.2"], "C004": ["05.6.1.1.E"], "C005": ["05.6.1.1.C"],
    "C006": ["05.6.1.9.9","05.6.1.9.A"], "C007": ["05.6.1.9.6","05.6.1.9.8"],
    "C008": ["05.6.1.1.8","05.6.1.1.F"], "C009": ["05.6.1.1.9"],
    "C010": ["05.6.1.1.B"], "C011": ["05.6.1.9.3"], "C012": ["05.5.2.2.2"],
    "C013": ["05.5.2.2.1"], "C014": ["05.6.1.1.3"], "C015": ["05.6.1.9.5"],
    # PERSONAL CARE
    "D021": ["13.1.2.0.4"], "D022": ["13.1.3.1.1","13.1.3.1.2"],
    "D023": ["13.1.3.2.1","13.1.3.2.2"], "D024": ["13.1.3.1.3"],
    "D025": ["13.1.3.1.4"], "D026": ["13.1.2.0.J","13.1.3.2.3","13.9.0.1.0"],
    # EDUCATION
    "E001": ["10.1.0.1.1","10.1.0.1.2"], "E002": ["10.1.0.2.1","10.1.0.2.2"],
    "E003": ["10.2.0.0.1","10.2.0.0.2"], "E004": ["10.3.0.0.1","10.3.0.0.2"],
    "E005": ["09.7.4.0.9","10.4.0.1.5","10.4.0.1.6"],
    "E006": ["10.4.0.1.8","10.4.0.1.9"],
    "E007": ["09.7.4.0.B","10.4.0.1.1","10.4.0.1.2","10.4.0.1.4"],
    "E008": ["13.3.0.1.1","13.3.0.1.2","13.3.0.1.3"],
    "E009": ["07.2.4.3.1","09.4.6.1.1","09.4.6.1.4","09.4.6.2.3",
             "09.7.4.0.A","10.5.0.9.1","10.5.0.9.2","10.5.0.9.3"],
    "E010": [], "E011": [], "E012": ["13.3.0.1.4"], "E013": ["07.3.2.3.0"],
    "E014": ["09.7.1.1.1"],
    "E015": ["10.1.0.1.4","10.1.0.2.4","10.2.0.0.4","10.3.0.0.4"],
    "E028": ["09.6.1.0.2","09.6.1.0.3"],
    "E029": ["09.4.6.1.2","11.1.1.1.4","11.1.1.1.5"],
    "E030": ["09.4.6.2.5","09.4.6.3.0"], "E031": ["09.4.7.0.0"],
    "E032": ["09.4.6.2.2","09.4.6.2.4"], "E033": ["08.3.5.0.2"],
    "E034": ["08.3.9.1.0","08.3.9.2.3","09.4.6.1.5","09.4.6.1.6",
             "09.6.9.0.4","08.3.9.2.1","09.6.1.0.4","09.6.2.0.1",
             "09.6.2.0.2","08.3.9.2.2","08.6.1.3","09.6.9.0.3"],
    # COMMUNICATIONS AND VEHICLES
    "F001": ["08.1.1.0.0","08.3.1.0.3"], "F002": ["08.1.2.0.0"],
    "F003": ["08.3.2.0.1"], "F004": ["08.3.1.0.4"],
    "F005": ["07.4.1.1.0","07.4.1.2.0","07.4.9.2.0"],
    "F006": ["08.3.3.1.2","08.3.3.1.3","08.3.5.0.1","08.3.9.9.0"],
    "F007": ["07.2.2.2.1"], "F008": ["07.2.2.2.2"],
    "F009": ["07.2.2.1.0","07.2.2.3.0"],
    "F010": ["07.2.2.4.0","07.2.1.3.4"], "F011": ["07.2.3.0.6"],
    # WOMEN'S CLOTHING
    "H068": ["03.1.2.2.1"], "H069": ["03.1.2.2.2"], "H070": ["03.1.2.2.3"],
    "H071": ["03.1.2.2.4"], "H072": ["03.1.2.2.6"], "H073": [], "H074": [],
    "H075": ["03.1.2.2.5"], "H076": ["03.1.2.2.7"],
    "H077": ["03.1.2.2.A"], "H078": ["03.1.2.2.B"], "H079": ["03.1.2.2.C"],
    "H080": ["03.1.2.2.8"], "H081": ["03.1.2.2.9"], "H082": ["03.1.3.2.0"],
    "H114": ["03.2.1.2.1"], "H115": ["03.2.1.2.5"], "H116": ["03.2.1.2.2"],
    "H117": ["03.2.1.2.3"], "H118": ["03.2.1.2.4"],
    # MEN'S CLOTHING
    "H056": ["03.1.2.1.1"], "H057": ["03.1.2.1.2"], "H058": ["03.1.2.1.3"],
    "H059": ["03.1.2.1.4"], "H061": ["03.1.2.1.5"], "H062": ["03.1.2.1.7"],
    "H063": ["03.1.2.1.8"], "H064": ["03.1.2.1.9"], "H065": ["03.1.2.1.6"],
    "H108": ["03.2.1.1.1"], "H109": ["03.2.1.1.5"],
    "H110": ["03.2.1.1.2"], "H111": ["03.2.1.1.3"], "H112": ["03.2.1.1.4"],
    "H123": ["13.2.9.1.1"], "H124": ["03.1.3.1.1","13.2.9.1.2"],
    "H126": ["13.2.9.1.9"], "H127": ["13.2.1.1.1"],
    # HEALTH
    "J002": ["06.3.0.1.4"], "J007": ["06.2.1.9.2","06.2.1.9.4"],
    "J009": ["06.1.1.1.B"], "J010": ["01.9.9.4"],
    "J011": ["06.4.1.0.3"], "J012": ["06.3.0.1.3"], "J013": ["06.2.1.9.8"],
    "J044": ["06.1.1.1.8"], "J043": ["06.4.2.0.1","06.2.3.2.2"],
    "J061": ["06.1.1.1.J"], "J062": ["06.2.1.9.5"], "J063": ["06.1.1.2.0"],
    "J065": ["06.1.3.1.2","06.1.3.2.0"],
    "J067": ["06.1.3.3.1","06.1.3.3.2"], "J068": ["06.1.4.0.1"],
    # HOUSEHOLD FURNISHINGS
    "I008": ["13.2.1.1.3"], "I009": ["05.4.0.3.1"],
    "I011": ["05.4.0.3.2","05.4.0.3.4"], "I012": ["05.5.2.1.1"],
    "I013": ["05.4.0.4.0"], "I014": ["05.1.1.1.1"], "I015": ["05.2.1.9.2"],
    "I016": ["05.2.1.2.3"], "I017": ["05.2.1.2.1"], "I018": ["05.2.1.2.4"],
    "I019": ["05.2.1.2.2"], "I020": ["05.2.1.3.1","05.4.0.3.3"],
    "I021": ["05.2.1.3.2"], "I022": ["05.1.1.4.5","05.2.1.1.1","05.2.1.1.2"],
    "I023": ["05.2.1.9.3","05.2.2.0.0"], "I025": ["05.6.1.9.1"],
    "I026": ["05.2.1.1.3"],
    # HOUSEHOLD APPLIANCES
    "K024": ["05.3.1.1.4","05.3.1.2.2","05.3.1.3.1","05.3.2.0.3",
             "05.3.2.2.1","05.3.2.0.5","05.3.1.3.2","05.3.2.0.2",
             "05.3.2.0.6","05.3.2.0.7","05.3.2.9.3"],
    "K025": ["05.3.3.0.1","05.3.3.0.2","05.3.3.0.3","05.5.3.0.2"],
    "K026": ["05.1.1.1.4"], "K027": ["05.1.1.4.3"], "K028": ["05.1.1.1.2"],
    "K030": ["05.1.1.1.5"], "K032": ["05.1.1.1.3"], "K034": ["05.1.1.2.1"],
    "K035": ["05.1.1.4.1"], "K036": ["05.1.1.1.6","05.3.1.9.0"],
    "K037": ["05.1.2.0.1"],
    # ELECTRONICS AND TELECOM
    "L004": ["08.1.4.0.A"], "L005": ["08.1.4.0.A"], "L006": ["08.1.4.0.8"],
    "L007": ["08.1.3.1.1","08.1.3.1.2","08.1.3.1.3","08.1.3.2.4"],
    "L013": ["08.1.4.0.3"],
    "L018": ["09.1.1.1.1","09.1.1.1.2","09.1.1.1.3"],
    "R001": ["08.3.2.0.1"],
    "R007": ["08.3.2.0.2"], "R008": ["08.3.3.1.1"], "R009": ["08.3.9.2.4"],
    "R010": ["08.3.4.0.1"],
    "R011": ["08.3.4.0.4","08.3.4.0.5","08.3.4.0.2","08.3.4.0.3"],
    # VEHICLES
    "M007": ["07.1.1.1.0","07.1.1.2.0"], "M008": ["07.1.1.1.0"],
    "M009": ["07.1.2.0.0"], "M010": ["07.1.3.0.0"],
    "M012": ["07.2.1.1.0"], "M013": ["07.2.1.2.2"], "M014": ["07.2.1.2.1"],
    "M015": ["07.2.1.3.3"], "M016": ["07.2.1.3.1","07.2.1.3.2"],
    "M017": ["07.2.3.0.1","07.2.3.0.3"],
    "M018": ["07.2.3.0.4","09.4.2.1.0","07.2.3.0.2","07.2.3.0.5"],
    # MISCELLANEOUS GOODS AND SERVICES
    "N001": ["09.6.3.0.0","12.2.1.0.0","12.2.9.9.0",
             "13.3.0.9.1","13.3.0.9.2","13.9.0.9.1"],
    "N002": ["13.2.9.1.A","13.9.0.9.2"],
    "N003": ["09.2.1.3.1","09.6.1.0.5","09.6.9.0.5","09.6.9.0.6"],
    "N013": ["13.9.0.9.0"],
    # HOUSING
    "G101": ["04.1.1.0.0"], "G009": ["04.5.1.0.0"],
    # FINANCIAL EXPENDITURES
    "Q001": ["14.1.1.1.1"], "Q002": ["14.1.1.1.2"], "Q003": ["14.1.1.1.3"],
    "Q004": ["14.1.1.1.4"], "Q005": ["14.1.1.1.5"], "Q006": ["05.1.1.4.4"],
    "Q007": ["14.1.1.1.7"], "Q008": ["14.1.1.1.8"], "Q009": ["14.1.1.1.9"],
    "Q010": ["14.1.1.1.A"], "Q011": ["14.1.1.1.B"], "Q012": ["14.1.1.1.C"],
    "Q013": ["14.1.1.1.D"], "Q015": ["14.1.1.1.E"],
}

# Expand to long format (one row per ENIGH code × CCIF class pair)
rows = []
for clave, ccif_list in CROSSWALK_OFICIAL.items():
    for ccif in ccif_list:
        rows.append({
            "clave_enigh": clave.strip(),
            "ccif":        ccif,
            "ccif_div":    ccif[:2]
        })
df_xwalk = pd.DataFrame(rows)

print(f"ENIGH codes in crosswalk  : {len(CROSSWALK_OFICIAL)}")
print(f"ENIGH–CCIF pairs (long)   : {len(df_xwalk)}")
print(f"CCIF divisions covered    : {sorted(df_xwalk['ccif_div'].unique())}")


## Section 4 — Expenditure Long Table and Crosswalk Merge

Household (`gastoshogar`) and individual (`gastospersona`) expenditure records are stacked into a single long-format table. Each row represents one expenditure item for one household in a given quarter.

Quarterly expenditure (`gasto_tri`) is used as the monetary variable, consistent with the ENIGH Seasonal survey design. Records with missing or zero values are excluded.

The expenditure table is then merged with the ENIGH–CCIF crosswalk (inner join), which restricts the analysis to spending items with a validated classification. Coverage is reported as both a code match rate and a share of total weighted pesos.


In [ ]:
# ── 4.1  Build unified expenditure long table ────────────────────────────────
df_gh["clave_gasto"] = df_gh["clave"].astype(str).str.strip()
df_gp["clave_gasto"] = df_gp["clave"].astype(str).str.strip()
df_gh["_monto"]      = pd.to_numeric(df_gh["gasto_tri"], errors="coerce")
df_gp["_monto"]      = pd.to_numeric(df_gp["gasto_tri"], errors="coerce")

df_gasto_long = pd.concat(
    [df_gh[["IDHOG", "clave_gasto", "_monto"]],
     df_gp[["IDHOG", "clave_gasto", "_monto"]]],
    ignore_index=True
).dropna(subset=["_monto"])
df_gasto_long = df_gasto_long[df_gasto_long["_monto"] > 0].reset_index(drop=True)

print(f"Expenditure records (after cleaning): {len(df_gasto_long):,}")
print(f"Unique ENIGH codes in data           : {df_gasto_long['clave_gasto'].nunique()}")


In [ ]:
# ── 4.2  Merge with CCIF crosswalk and report coverage ───────────────────────
df_mapped = df_gasto_long.merge(
    df_xwalk, left_on="clave_gasto", right_on="clave_enigh", how="inner"
)

n_total  = df_gasto_long["clave_gasto"].nunique()
n_mapped = df_mapped["clave_gasto"].nunique()
pct_pesos = df_mapped["_monto"].sum() / df_gasto_long["_monto"].sum() * 100

print(f"Codes matched    : {n_mapped} / {n_total}  ({n_mapped/n_total:.1%})")
print(f"Spending covered : {pct_pesos:.1f}% of total pesos")
print(f"Rows in table    : {len(df_mapped):,}")


## Section 5 — Master Dataset and Survey Weights

The master dataset (`master2`) combines quarterly household expansion factors (`factor_t`) with headship typology. The quarterly weight `factor_t` is used throughout because the ENIGH Seasonal survey is designed for quarterly, not annual, population estimation.

This table serves as the basis for all weighted aggregations in Sections 6 and 7.


In [ ]:
# ── 5.1  Build master2: quarterly weights merged with headship typology ────────
#
# factor_t: quarterly household expansion factor (INEGI ENIGH Seasonal design).
# All weighted means and totals use factor_t to produce population-representative estimates.

master2 = (
    df_hog[["IDHOG", "factor_t"]].copy()
    .assign(factor_t=lambda d: pd.to_numeric(d["factor_t"], errors="coerce"))
    .merge(
        headship[["IDHOG", "sexo_jefe", "tipo_jefatura", "tiene_pareja"]],
        on="IDHOG", how="left"
    )
)

print(f"Master dataset: {master2.shape[0]:,} households")
print("\nDistribution of headship types (weighted):")
for cat, n in master2["tipo_jefatura"].value_counts(dropna=False).items():
    print(f"  {cat:<35} {n:>6,}")


## Section 6 — G-CPI Expenditure Weights by CCIF Division

For each CCIF division and each headship group (HOMBRE / MUJER), the expenditure weight is computed as:

$$w_{i,g} = \frac{\sum_{h \in g} gasto_{tri,h,i} \times factor\_t_h}{\sum_{h \in g} \sum_{j} gasto_{tri,h,j} \times factor\_t_h}$$

where $g$ indexes the headship group, $h$ indexes households, $i$ indexes CCIF divisions, and $j$ runs over all classified divisions.

These weights constitute the gender-differentiated basket for the G-CPI Laspeyres-type price index computed in Section 7.


In [ ]:
# ── 6.1  Compute weighted expenditure shares by CCIF division and headship ─────
analysis_df = (
    df_mapped
    .merge(master2[["IDHOG", "factor_t", "sexo_jefe"]], on="IDHOG", how="inner")
    .dropna(subset=["factor_t", "sexo_jefe"])
)

# Total weighted expenditure per headship group
tot = (
    analysis_df
    .groupby("sexo_jefe")
    .apply(lambda g: (g["_monto"] * g["factor_t"]).sum())
    .rename("total_wgasto").reset_index()
)

# Weighted expenditure by division × headship group
by_div = (
    analysis_df
    .groupby(["sexo_jefe", "ccif_div"])
    .apply(lambda g: (g["_monto"] * g["factor_t"]).sum())
    .rename("wgasto").reset_index()
    .merge(tot, on="sexo_jefe")
)
by_div["peso"] = by_div["wgasto"] / by_div["total_wgasto"]

by_div.to_csv(OUT_DIR / "gcpi_weights_division_sexo_jefe.csv", index=False)
print("Saved: outputs/gcpi_weights_division_sexo_jefe.csv")

pivot_weights = (
    by_div.pivot_table(index="ccif_div", columns="sexo_jefe", values="peso")
    .round(4).sort_index()
)
print("\nG-CPI expenditure weights by CCIF division:")
print(pivot_weights.to_string())


In [ ]:
# ── 6.2  Identify categories with higher female vs male relative weight ─────────
DIV_NAMES = {
    "01": "Alimentos y bebidas no alcohólicas",
    "02": "Bebidas alcohólicas y tabaco",
    "03": "Vestido y calzado",
    "04": "Vivienda y servicios",
    "05": "Muebles y mantenimiento del hogar",
    "06": "Salud",
    "07": "Transporte",
    "08": "Comunicación",
    "09": "Recreación y cultura",
    "10": "Educación",
    "11": "Restaurantes y hoteles",
    "12": "Bienes y servicios diversos",
    "13": "Cuidado personal y otros servicios",
    "14": "Erogaciones financieras",
}

pv = (
    by_div.pivot_table(index="ccif_div", columns="sexo_jefe", values="peso")
    .fillna(0).reset_index()
)
pv["diff_M_minus_H"]  = pv.get("MUJER", 0) - pv.get("HOMBRE", 0)
pv["high_female"]     = pv["diff_M_minus_H"] > 0
pv["division_nombre"] = pv["ccif_div"].map(DIV_NAMES).fillna("Other")
pv = pv.sort_values("diff_M_minus_H", ascending=False)

pv.to_csv(OUT_DIR / "gcpi_high_female_consumption.csv", index=False)

print("Divisions with higher relative weight in female-headed households:")
print(pv[pv["high_female"]][["ccif_div","division_nombre","HOMBRE","MUJER","diff_M_minus_H"]].to_string(index=False))
print("\nDivisions with higher relative weight in male-headed households:")
print(pv[~pv["high_female"]][["ccif_div","division_nombre","HOMBRE","MUJER","diff_M_minus_H"]].to_string(index=False))


In [ ]:
# ── 6.3  Dumbbell chart: expenditure structure comparison ─────────────────────
#
# Each point represents the expenditure weight allocated to a CCIF division
# by male-headed (blue) and female-headed (red) households.
# Red connectors indicate divisions where female-headed households allocate
# a larger share; blue connectors indicate the reverse.

pv_plot = pv.sort_values("diff_M_minus_H")
y   = np.arange(len(pv_plot))
x_h = pv_plot["HOMBRE"].to_numpy() if "HOMBRE" in pv_plot.columns else np.zeros(len(pv_plot))
x_m = pv_plot["MUJER"].to_numpy()  if "MUJER"  in pv_plot.columns else np.zeros(len(pv_plot))
labels = pv_plot["ccif_div"].astype(str) + " — " + pv_plot["division_nombre"]

fig, ax = plt.subplots(figsize=(13, 8))
for i in range(len(pv_plot)):
    color = "#E57373" if x_m[i] > x_h[i] else "#64B5F6"
    ax.plot([x_h[i], x_m[i]], [y[i], y[i]], color=color, lw=2.5, alpha=0.7)
ax.scatter(x_h, y, color="#1565C0", zorder=5, s=80, label="Male-headed household")
ax.scatter(x_m, y, color="#C62828", zorder=5, s=80, label="Female-headed household")
ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel("Expenditure weight (share of total classified spending)", fontsize=10)
ax.set_title(
    "G-CPI: Expenditure Structure by CCIF Division\nMale-headed vs Female-headed Households",
    fontsize=12, fontweight="bold"
)
ax.legend(fontsize=10)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "gcpi_dumbbell_divisiones_ccif.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/gcpi_dumbbell_divisiones_ccif.png")


In [ ]:
# ── 6.4  Export final weight table ────────────────────────────────────────────
weight_table = pv[["ccif_div", "division_nombre", "HOMBRE", "MUJER",
                    "diff_M_minus_H", "high_female"]].copy()
weight_table.columns = ["ccif_div", "nombre", "peso_hombre", "peso_mujer",
                         "diff_mujer_menos_hombre", "high_female"]
weight_table.to_csv(OUT_DIR / "gcpi_weight_table_FINAL.csv", index=False)

print("Saved: outputs/gcpi_weight_table_FINAL.csv")
print("\nFinal G-CPI weight table:")
print(weight_table.to_string(index=False))


## Section 7 — G-CPI Inflation Series and Gender Gap

The G-CPI is constructed as a Laspeyres-type price index using the gender-differentiated expenditure weights from Section 6 and official INPC division-level price indices published by INEGI.

**Price index source:** INPC, base 2nd fortnight of July 2018 = 100, CCIF 2018 monthly series, downloaded from INEGI BIE (January 2026).  
**File:** `INDICE 2022_2025.csv`

The gender-specific price index for group $g$ at time $t$ is:

$$G\text{-}CPI_g(t) = \frac{\sum_i w_{i,g} \cdot P_i(t)}{\sum_i w_{i,g}}$$

Year-over-year inflation and the inflation gap are then defined as:

$$\pi_g(t) = \left(\frac{G\text{-}CPI_g(t)}{G\text{-}CPI_g(t-12)} - 1\right) \times 100$$

$$\text{Gap}(t) = \pi_{\text{Mujer}}(t) - \pi_{\text{Hombre}}(t)$$


In [ ]:
# ── 7.1  Parse INPC division-level price indices ──────────────────────────────
#
# The INEGI CSV export contains 10 metadata rows before the data begins.
# Column positions map directly to CCIF divisions 01–13 (columns 2–14).

indice = pd.read_csv(
    "INDICE 2022_2025.csv",
    encoding="latin-1",
    skiprows=10,
    header=None,
    names=[
        "fecha", "inpc_general",
        "div_01","div_02","div_03","div_04","div_05",
        "div_06","div_07","div_08","div_09","div_10",
        "div_11","div_12","div_13"
    ]
)

# Retain only rows that match the "mes-YY" date pattern
indice = indice[indice["fecha"].astype(str).str.match(r"^[a-zA-Z]{3}-\d{2}$")].copy()

# Map Spanish month abbreviations to English for datetime parsing
MONTH_MAP = {
    "ene":"Jan","feb":"Feb","mar":"Mar","abr":"Apr","may":"May","jun":"Jun",
    "jul":"Jul","ago":"Aug","sep":"Sep","oct":"Oct","nov":"Nov","dic":"Dec"
}

def parse_fecha(s):
    s = str(s).strip().lower()
    for sp, en in MONTH_MAP.items():
        s = s.replace(sp, en)
    return pd.to_datetime(s, format="%b-%y")

indice["fecha"] = indice["fecha"].apply(parse_fecha)

div_cols = [f"div_{i:02d}" for i in range(1, 14)]
for c in ["inpc_general"] + div_cols:
    indice[c] = pd.to_numeric(indice[c], errors="coerce")

indice = indice.sort_values("fecha").reset_index(drop=True)

print(f"{len(indice)} monthly observations")
print(f"{indice['fecha'].min().strftime('%b %Y')} → {indice['fecha'].max().strftime('%b %Y')}")
print(indice[["fecha","inpc_general","div_01","div_07","div_08"]].to_string(index=False))


In [ ]:
# ── 7.2  Align price indices with G-CPI weights ───────────────────────────────
weights = pd.read_csv("outputs/gcpi_weight_table_FINAL.csv")
weights["ccif_div_int"] = pd.to_numeric(weights["ccif_div"], errors="coerce").astype("Int64")
weights["indice_col"]   = weights["ccif_div_int"].map({i: f"div_{i:02d}" for i in range(1, 14)})

print("Division-to-column mapping and weights:")
print(weights[["ccif_div","nombre","indice_col","peso_hombre","peso_mujer"]].to_string(index=False))

# Build long table: one row per period × division
rows = []
for _, w in weights.dropna(subset=["indice_col"]).iterrows():
    col = w["indice_col"]
    if col in indice.columns:
        tmp = indice[["fecha", col]].copy()
        tmp.columns = ["fecha", "indice_precio"]
        tmp["ccif_div"]    = int(w["ccif_div_int"])
        tmp["peso_hombre"] = float(w["peso_hombre"]) if pd.notna(w["peso_hombre"]) else 0.0
        tmp["peso_mujer"]  = float(w["peso_mujer"])  if pd.notna(w["peso_mujer"])  else 0.0
        rows.append(tmp)

indice_long = pd.concat(rows, ignore_index=True).dropna(subset=["indice_precio"])

print(f"\nLong index table: {len(indice_long):,} rows")
print(f"Divisions : {sorted(indice_long['ccif_div'].unique())}")
print(f"Periods   : {indice_long['fecha'].nunique()}")


In [ ]:
# ── 7.3  Compute G-CPI series and year-over-year inflation rates ─────────────
gcpi_hombre = (
    indice_long.groupby("fecha", as_index=False)
    .apply(lambda g: pd.Series({
        "gcpi_hombre": (g["indice_precio"] * g["peso_hombre"]).sum() / g["peso_hombre"].sum()
    }), include_groups=False)
)
gcpi_mujer = (
    indice_long.groupby("fecha", as_index=False)
    .apply(lambda g: pd.Series({
        "gcpi_mujer": (g["indice_precio"] * g["peso_mujer"]).sum() / g["peso_mujer"].sum()
    }), include_groups=False)
)

gcpi = (
    gcpi_hombre.merge(gcpi_mujer, on="fecha")
    .sort_values("fecha").reset_index(drop=True)
)

# 12-month year-over-year inflation rate and gender gap
gcpi["inf_hombre"]    = gcpi["gcpi_hombre"].pct_change(12) * 100
gcpi["inf_mujer"]     = gcpi["gcpi_mujer"].pct_change(12)  * 100
gcpi["gap_inflacion"] = gcpi["inf_mujer"] - gcpi["inf_hombre"]

gcpi.to_csv(OUT_DIR / "gcpi_series_inflacion.csv", index=False)
print("Saved: outputs/gcpi_series_inflacion.csv\n")
print(gcpi.dropna().to_string(index=False))


In [ ]:
# ── 7.4  Visualization: inflation rates and gender gap ───────────────────────
#
# Top panel: annual inflation rate for male- and female-headed households.
# Shaded areas highlight periods where one group bears higher inflation.
# Bottom panel: inflation gap (Mujer − Hombre) in percentage points.
# Red shading indicates periods where female-headed households face higher inflation;
# blue shading indicates the reverse.

gcpi_plot = gcpi.dropna()
fig, axes = plt.subplots(2, 1, figsize=(13, 10), sharex=True)

# Top panel
ax1 = axes[0]
ax1.plot(gcpi_plot["fecha"], gcpi_plot["inf_hombre"],
         color="#1565C0", lw=2.5, label="Male-headed household")
ax1.plot(gcpi_plot["fecha"], gcpi_plot["inf_mujer"],
         color="#C62828", lw=2.5, label="Female-headed household")
ax1.fill_between(gcpi_plot["fecha"],
                 gcpi_plot["inf_hombre"], gcpi_plot["inf_mujer"],
                 where=gcpi_plot["inf_mujer"] >= gcpi_plot["inf_hombre"],
                 alpha=0.15, color="#C62828")
ax1.fill_between(gcpi_plot["fecha"],
                 gcpi_plot["inf_hombre"], gcpi_plot["inf_mujer"],
                 where=gcpi_plot["inf_mujer"] < gcpi_plot["inf_hombre"],
                 alpha=0.15, color="#1565C0")
ax1.set_ylabel("Annual inflation rate (%)", fontsize=11)
ax1.set_title("G-CPI: Inflation by Household Headship Type\n(Base: 2nd fortnight July 2018 = 100)",
              fontsize=12, fontweight="bold")
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=6))

# Bottom panel
ax2 = axes[1]
ax2.axhline(0, color="gray", lw=1.2, ls="--", alpha=0.7)
ax2.fill_between(gcpi_plot["fecha"], gcpi_plot["gap_inflacion"], 0,
                 where=gcpi_plot["gap_inflacion"] > 0,
                 color="#C62828", alpha=0.4, label="Women face higher inflation")
ax2.fill_between(gcpi_plot["fecha"], gcpi_plot["gap_inflacion"], 0,
                 where=gcpi_plot["gap_inflacion"] <= 0,
                 color="#1565C0", alpha=0.4, label="Men face higher inflation")
ax2.plot(gcpi_plot["fecha"], gcpi_plot["gap_inflacion"], color="#333333", lw=1.5)
ax2.set_ylabel("Gap (Female − Male, pp)", fontsize=11)
ax2.set_title("G-CPI Inflation Gap by Household Headship", fontsize=12, fontweight="bold")
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=6))

plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(OUT_DIR / "gcpi_inflation_gap_series.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/gcpi_inflation_gap_series.png")


## Section 8 — Summary Statistics and Output for Forecasting

Descriptive statistics for the inflation gap series are reported below. The output file `gap_series_para_forecast.csv` contains the full monthly series and serves as input for the forecasting models (SARIMA, VAR, and machine learning approaches) developed in a separate notebook.


In [ ]:
# ── 8.1  Descriptive statistics for the G-CPI inflation gap ──────────────────
s = gcpi_plot.copy()

print("=" * 60)
print("  G-CPI SUMMARY STATISTICS")
print("=" * 60)
print(f"  Period                        : {s['fecha'].min().strftime('%b %Y')} → {s['fecha'].max().strftime('%b %Y')}")
print(f"  Monthly observations          : {len(s)}")
print()
print(f"  Mean inflation  Male-headed   : {s['inf_hombre'].mean():.3f}%")
print(f"  Mean inflation  Female-headed : {s['inf_mujer'].mean():.3f}%")
print()
print(f"  Mean gap (Female − Male)      : {s['gap_inflacion'].mean():.4f} pp")
print(f"  Std. deviation of gap         : {s['gap_inflacion'].std():.4f} pp")
print(f"  Maximum gap (women pay more)  : {s['gap_inflacion'].max():.4f} pp  [{s.loc[s['gap_inflacion'].idxmax(),'fecha'].strftime('%b %Y')}]")
print(f"  Minimum gap (men pay more)    : {s['gap_inflacion'].min():.4f} pp  [{s.loc[s['gap_inflacion'].idxmin(),'fecha'].strftime('%b %Y')}]")
print()
print(f"  % periods Female > Male       : {(s['gap_inflacion'] > 0).mean()*100:.1f}%")
print(f"  % periods Male > Female       : {(s['gap_inflacion'] < 0).mean()*100:.1f}%")
print("=" * 60)
print()
print("Full descriptive statistics:")
print(s[["inf_hombre","inf_mujer","gap_inflacion"]].describe().round(4).to_string())


In [ ]:
# ── 8.2  Export series for forecasting notebook ───────────────────────────────
s[["fecha","gcpi_hombre","gcpi_mujer","inf_hombre","inf_mujer","gap_inflacion"]].to_csv(
    OUT_DIR / "gap_series_para_forecast.csv", index=False
)
print("Saved: outputs/gap_series_para_forecast.csv")
print("This file is used as input for the SARIMA, VAR, and ML forecasting models.")
